# Gemma 4 Study Companion — Kaggle como servidor (API + ngrok)

Este notebook convierte el backend original (que estaba pensado como app de Gradio) en un **servidor HTTP con FastAPI**, expuesto a internet con **ngrok**, para que puedas consumirlo desde cualquier cliente externo (tu propia app, un script, Postman, etc.) mientras Kaggle mantiene la GPU y el modelo Gemma 4 cargados en memoria.

## Qué cambia respecto al notebook original
- Se quita Gradio como interfaz. Ya no hay UI dentro del notebook.
- Toda la lógica de negocio (perfil, temario, lecciones, feedback, apuntes por imagen, audio, HTML interactivo, visualización con Plotly, favoritos, progreso, exportación) se expone como **endpoints REST**.
- El notebook original acumuló 26 iteraciones ("v9", "v13", "v26"...) con varias funciones duplicadas y algunas rotas (el flujo de "audio nativo a Gemma" de la v26 llama a funciones que nunca llegaron a definirse, como `generate_lesson_ui` con la firma nueva o `gemma_response_from_audio`). Este notebook parte de las piezas que sí funcionan (Gemma para texto/imagen, Whisper para audio→texto) y las deja consistentes en un solo lugar.
- El audio de entrada (perfil, transcripción) se resuelve con **Whisper** (ya estaba integrado en el notebook original) y luego el texto transcrito va a Gemma — es el mismo patrón robusto que ya usabas para el perfil por voz.

## Requisitos antes de correr
1. **GPU activada** en el notebook (Settings → Accelerator → GPU).
2. Dos secretos en **Add-ons → Secrets**:
   - `HFTOKEN`: tu token de Hugging Face con acceso a Gemma.
   - `NGROK_TOKEN`: tu token de [ngrok.com](https://ngrok.com) (cuenta gratuita alcanza).
3. Ejecuta las celdas en orden. La última celda deja el servidor corriendo en background con una URL pública tipo `https://xxxx.ngrok-free.app`.
4. Mientras el notebook/kernel siga activo (con "Save & Run All" en modo interactivo, o con la sesión abierta), el servidor sigue disponible.

## 1. Dependencias

In [1]:
!apt-get update -y > /dev/null 2>&1
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q --upgrade git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade accelerate bitsandbytes huggingface_hub gTTS pydantic json-repair pymupdf pillow sentencepiece plotly kaleido
!pip install -q --upgrade fastapi "uvicorn[standard]" python-multipart pyngrok nest_asyncio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 10.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 57.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 81.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -q --force-reinstall --no-cache-dir "numpy>=2.0" "scipy"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 227.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 276.0 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is

In [ ]:
import os
print("🔁 Reiniciando el runtime para aplicar numpy/scipy/transformers limpios...")
os.kill(os.getpid(), 9)

## 2. Configuración, modelo Gemma 4 y ASR (Whisper)

Carga el modelo cuantizado en 4-bit y el pipeline de transcripción, igual que en el notebook original. También valida que `HFTOKEN` y `NGROK_TOKEN` existan como secretos.

In [1]:
import os, json, re, textwrap, datetime, threading, uuid
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Literal

import torch
import fitz
from PIL import Image, ImageDraw
from gtts import gTTS
from pydantic import BaseModel, Field
from json_repair import repair_json
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, pipeline
import plotly.graph_objects as go

In [2]:
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = ROOT / 'study_companion_server'
ASSETS = OUT / 'assets'
MEDIA = OUT / 'media'
HTML = OUT / 'html'
DATA = OUT / 'data'
UPLOADS = OUT / 'uploads'
EXPORT_DIR = DATA / 'generated_exports'
for p in [OUT, ASSETS, MEDIA, HTML, DATA, UPLOADS, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = DATA / 'user_profile.json'
FAVORITES_PATH = DATA / 'favorites.json'
PROGRESS_PATH = DATA / 'progress.json'
LESSONS_PATH = DATA / 'lesson_memory.json'
NOTE_IMAGES_PATH = DATA / 'note_images.json'

for path, default in [
    (FAVORITES_PATH, []),
    (PROGRESS_PATH, {}),
    (LESSONS_PATH, {}),
    (NOTE_IMAGES_PATH, {}),
]:
    if not path.exists():
        path.write_text(json.dumps(default, ensure_ascii=False, indent=2), encoding='utf-8')


def read_json(path: Path, default):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')


# --- Credenciales ---
# HFTOKEN: token de Hugging Face para descargar Gemma.
# NGROK_TOKEN: token de ngrok para exponer el servidor.
# Configúralos en Kaggle -> Add-ons -> Secrets, o como variable de entorno.

def _get_secret(name: str) -> Optional[str]:
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


HFTOKEN = _get_secret('HFTOKEN')
if not HFTOKEN:
    raise RuntimeError('Configura HFTOKEN en Kaggle Add-ons > Secrets.')

NGROK_TOKEN = _get_secret('NGROK_TOKEN')
if not NGROK_TOKEN:
    raise RuntimeError('Configura NGROK_TOKEN en Kaggle Add-ons > Secrets (crea uno gratis en ngrok.com).')

login(token=HFTOKEN)
MODEL_ID = os.getenv('GEMMA_MODEL_ID', 'google/gemma-4-E4B-it')
assert torch.cuda.is_available(), 'Activa GPU en Kaggle (Settings > Accelerator).'

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HFTOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HFTOKEN
).eval()

asr_pipe = pipeline('automatic-speech-recognition', model='openai/whisper-small', device=0 if torch.cuda.is_available() else -1)

print('Gemma listo:', MODEL_ID)
print('Whisper listo')

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Gemma listo: google/gemma-4-E4B-it
Whisper listo


## 3. Helpers núcleo

`gemma_generate` (texto), `gemma_multimodal_generate` (texto + imágenes), extracción robusta de JSON y transcripción de audio.

In [3]:
def gemma_generate(system_prompt: str, user_prompt: str, max_new_tokens: int = 2600, temperature: float = 0.35) -> str:
    messages = [{
        'role': 'user',
        'content': [{'type': 'text', 'text': f'SYSTEM:\n{system_prompt}\n\nUSER:\n{user_prompt}'}]
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    ).to(model.device, dtype=model.dtype if hasattr(model, 'dtype') else torch.bfloat16)
    input_len = inputs['input_ids'].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
    return processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def gemma_multimodal_generate(system_prompt: str, user_text: str, image_paths: List[str], max_new_tokens: int = 1800, temperature: float = 0.3) -> str:
    content = []
    for path in image_paths:
        content.append({'type': 'image', 'image': str(path)})
    content.append({'type': 'text', 'text': f'SYSTEM:\n{system_prompt}\n\nUSER:\n{user_text}'})
    messages = [{'role': 'user', 'content': content}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    ).to(model.device, dtype=model.dtype if hasattr(model, 'dtype') else torch.bfloat16)
    input_len = inputs['input_ids'].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
    return processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def extract_json_block(raw_text: str, required_keys: Optional[List[str]] = None, debug_label: str = '') -> dict:
    """Extrae un bloque JSON de la respuesta de Gemma de forma tolerante a texto
    explicativo antes/despues, bloques ```json ... ``` en cualquier posicion (no
    solo al inicio) y JSON casi valido (comas colgantes, comillas simples, etc.
    via json_repair).

    A proposito NO devuelve un dict vacio si falla: lanza ValueError con la salida
    cruda de Gemma incluida en el mensaje. Si devolvieramos {} aqui, Pydantic
    rellenaria cada campo con su valor por defecto (ej. nombre='Estudiante') sin
    ningun error visible, que es exactamente el bug de "no extrae nada" que
    reportaste: fallaba en silencio y el default lo disimulaba.
    """
    original = raw_text
    text = raw_text.strip()
    text = re.sub(r'```json', '', text, flags=re.IGNORECASE)
    text = text.replace('```', '')
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        print(f"[extract_json_block:{debug_label}] Gemma no devolvio un bloque {{...}} reconocible. Salida cruda:\n{original[:800]}")
        raise ValueError(f"Gemma no devolvio JSON reconocible ({debug_label}). Salida cruda: {original[:300]}")

    candidate = text[start:end + 1]
    try:
        data = json.loads(candidate)
    except Exception:
        try:
            data = json.loads(repair_json(candidate))
        except Exception as e:
            print(f"[extract_json_block:{debug_label}] JSON irreparable: {e}\nCandidato:\n{candidate[:800]}")
            raise ValueError(f"JSON irreparable en la respuesta de Gemma ({debug_label}): {e}")

    if not isinstance(data, dict):
        raise ValueError(f"Se esperaba un objeto JSON y se obtuvo {type(data)} ({debug_label}).")

    if required_keys:
        faltantes = [k for k in required_keys if not str(data.get(k) or '').strip() and not data.get(k)]
        if faltantes:
            print(f"[extract_json_block:{debug_label}] Gemma dejo vacios/omitio estos campos clave: {faltantes}. "
                  f"JSON recibido: {json.dumps(data, ensure_ascii=False)[:800]}")

    return data


def transcribe_audio_safe(audio_path: Optional[str]) -> str:
    if not audio_path:
        return ''
    try:
        result = asr_pipe(audio_path, chunk_length_s=20, batch_size=4)
        return (result.get('text') or '').strip()
    except Exception as e:
        return f'[error_transcribiendo_audio] {e}'

## 4. Modelos de datos (Pydantic)

Los mismos modelos de dominio del notebook original: perfil, temario, lección, micro-retos, ejemplos.

In [4]:
class LearnerProfile(BaseModel):
    nombre: str = 'Estudiante'
    objetivo: str = ''
    nivel_autopercibido: Literal['principiante', 'intermedio', 'avanzado'] = 'principiante'
    estilo_preferido: str = ''
    formato_preferido: str = ''
    duracion_sesion: str = ''
    puntos_de_bloqueo: List[str] = Field(default_factory=list)
    motivacion: str = ''
    tono_preferido: str = 'amigable'
    preferencias_extra: str = ''
    # NUEVO: permite personalizar la experiencia (ej. adulto mayor -> interfaces simples).
    # Todos los campos de arriba ahora son opcionales para que un audio corto o incompleto
    # igual genere un perfil utilizable; lo unico que realmente importa es el nombre.
    grupo_etario: Literal['nino', 'adolescente', 'adulto', 'adulto_mayor'] = 'adulto'
    necesidades_especiales: str = ''


class Subtema(BaseModel):
    nombre: str
    prerrequisitos: List[str] = Field(default_factory=list)
    errores_comunes: List[str] = Field(default_factory=list)
    tipo_de_concepto: Literal['declarativo', 'procedimental', 'dinamico'] = 'declarativo'


class TemaCurricular(BaseModel):
    tema: str
    subtemas: List[Subtema] = Field(default_factory=list)


class Materia(BaseModel):
    materia: str
    unidad: str
    temas: List[TemaCurricular] = Field(default_factory=list)


class EjemploNivel(BaseModel):
    nivel: Literal['principiante', 'intermedio', 'avanzado']
    enunciado: str
    pista: str
    respuesta: str


class MicroReto(BaseModel):
    pregunta: str
    opciones: List[str]
    respuesta_correcta: int
    explicacion: str


class Leccion(BaseModel):
    titulo: str
    tema_padre: str
    subtema: str
    nivel_tecnico: Literal['principiante', 'intermedio', 'avanzado'] = 'principiante'
    enfoque_usuario: str = ''
    conceptos_previos: List[str] = Field(default_factory=list)
    mapa_conocimiento: List[str] = Field(default_factory=list)
    explicacion_paso_a_paso: List[str] = Field(default_factory=list)
    pausa_reflexion: str = ''
    glosario: Dict[str, str] = Field(default_factory=dict)
    micro_retos: List[MicroReto] = Field(default_factory=list)
    ejemplos: List[EjemploNivel] = Field(default_factory=list)
    historia_base: str = ''
    guion_audio: str = ''
    latex_clave: List[str] = Field(default_factory=list)
    prompt_ilustracion: str = ''
    html_prompt: str = ''
    recomendacion_visual: Literal['imagen', 'html', 'combinada'] = 'html'
    visual_tipo: Literal['integral', 'complex', 'function', 'geometry', 'default'] = 'default'
    liked: bool = False
    version: int = 1
    feedback_historial: List[str] = Field(default_factory=list)


@dataclass
class PageBlock:
    page: int
    text: str
    image_paths: List[str] = field(default_factory=list)

## 5. Prompts de Gemma 4

Se mantienen los prompts de sistema originales (perfil, temario, lección, feedback) y se añade el de lectura de apuntes por imagen, que antes estaba suelto en otra celda.

In [5]:
PROFILE_SYSTEM = r'''
Eres un disenador de perfiles de aprendizaje. Recibes la transcripcion de una entrevista o audio de un
estudiante (puede ser corta, informal, incompleta o con errores de transcripcion).

TU UNICA SALIDA debe ser un objeto JSON. No escribas nada antes ni despues del JSON: nada de saludos,
explicaciones ni frases como "Aqui esta el perfil:". No copies el esquema con valores de ejemplo tipo "..."
en la respuesta final: cada campo debe llevar el valor real que extrajiste o infieriste, o "" / [] si de
verdad no hay informacion para ese campo.

PRIORIDAD MAXIMA: identifica el nombre del estudiante o como quiere que le llamen. Casi siempre aparece
cerca del inicio del texto ("me llamo...", "soy...", "mi nombre es...", "llamame..."). Solo si el estudiante
JAMAS menciona su nombre en ningun punto del texto, usa el valor "Estudiante" en el campo "nombre".

Para el resto de los campos: extrae lo explicito y, si no esta explicito, infierelo de forma conservadora del
contexto (tono, vocabulario, tema mencionado). Todos los campos deben existir en el JSON aunque su valor sea
una cadena u lista vacia; nunca inventes datos muy especificos sin base en el texto.

Presta especial atencion a pistas de edad o etapa de vida (ej. "soy jubilado", "tengo 70 anios", "voy en
la secundaria", "soy un nino", "soy abuelo y quiero ayudar a mis nietos", "soy estudiante", "no estudie mucho"). 
Usa esta informacion para inferir "grupo_etario". 
Si el estudiante parece un adulto mayor, o pide explicitamente algo simple y facil de usar, marca grupo_etario como "adulto_mayor" y describe eso en
"necesidades_especiales" (ej. "prefiere letras grandes y poca navegacion, poca experiencia con tecnologia, explicar conceptos basicos desde cero sin asumir conocimientos previos").
Si es una persona que dice "no se nada del tema", "soy principiante", "no estudie", "soy abuelo",
asegurate de marcar su nivel como "principiante" y agregar en "necesidades_especiales" instrucciones sobre 
usar un lenguaje libre de jerga tecnica, muy paso a paso, muy comprensivo y paciente. 
Para estudiantes avanzados o jovenes que dominan la tecnologia, ajusta el perfil para retos mayores.

Formato EXACTO de salida (JSON puro, sin markdown, sin comentarios, sin texto adicional):
{
  "nombre": "...",
  "objetivo": "...",
  "nivel_autopercibido": "principiante|intermedio|avanzado",
  "estilo_preferido": "...",
  "formato_preferido": "...",
  "duracion_sesion": "...",
  "puntos_de_bloqueo": ["..."],
  "motivacion": "...",
  "tono_preferido": "...",
  "preferencias_extra": "...",
  "grupo_etario": "nino|adolescente|adulto|adulto_mayor",
  "necesidades_especiales": "..."
}

Ejemplo de referencia 1:
Entrevista: "Hola, soy Marco, tengo 16 anios y voy en la prepa. Quiero entender mejor calculo..."
Salida esperada:
{
  "nombre": "Marco",
  "objetivo": "Entender mejor calculo",
  "nivel_autopercibido": "intermedio",
  "estilo_preferido": "visual, con ejemplos",
  "formato_preferido": "videos cortos y ejemplos visuales",
  "duracion_sesion": "",
  "puntos_de_bloqueo": ["limites"],
  "motivacion": "",
  "tono_preferido": "amigable",
  "preferencias_extra": "",
  "grupo_etario": "adolescente",
  "necesidades_especiales": ""
}

Ejemplo de referencia 2:
Entrevista: "Soy don Juan, tengo 68, soy abuelo y quiero aprender a usar esto de la compu, la verdad no estudie mucho y me cuesta."
Salida esperada:
{
  "nombre": "Juan",
  "objetivo": "Aprender computacion basica",
  "nivel_autopercibido": "principiante",
  "estilo_preferido": "paso a paso, muy simple",
  "formato_preferido": "muy visual y sin distracciones",
  "duracion_sesion": "",
  "puntos_de_bloqueo": ["tecnologia", "falta de experiencia"],
  "motivacion": "aprender algo nuevo como abuelo",
  "tono_preferido": "paciente y respetuoso",
  "preferencias_extra": "",
  "grupo_etario": "adulto_mayor",
  "necesidades_especiales": "Interfaz simple, letras grandes, sin jerga tecnica, muy paciente."
}
'''

CURRICULUM_SYSTEM = r'''
Eres un estructurador de temarios universitarios. Devuelve SOLO JSON valido con esta forma:
{
  "materia": "...",
  "unidad": "...",
  "temas": [
    {
      "tema": "...",
      "subtemas": [
        {
          "nombre": "...",
          "prerrequisitos": ["..."],
          "errores_comunes": ["..."],
          "tipo_de_concepto": "declarativo|procedimental|dinamico"
        }
      ]
    }
  ]
}
'''

LESSON_SYSTEM = r'''
Eres un tutor personal de matematicas. Recibes un subtema, un perfil de estudiante y un enfoque adicional.
Devuelve SOLO JSON valido con esta forma:
{
  "titulo": "...",
  "conceptos_previos": ["..."],
  "mapa_conocimiento": ["..."],
  "explicacion_paso_a_paso": ["..."],
  "pausa_reflexion": "...",
  "glosario": {"termino": "definicion"},
  "micro_retos": [
    {"pregunta": "...", "opciones": ["...","...","...","..."], "respuesta_correcta": 0, "explicacion": "..."}
  ],
  "ejemplos": [
    {"nivel": "principiante", "enunciado": "...", "pista": "...", "respuesta": "..."}
  ],
  "historia_base": "...",
  "guion_audio": "...",
  "latex_clave": ["\\\\(a+b\\\\)", "\\\\[x^2\\\\]"],
  "prompt_ilustracion": "...",
  "html_prompt": "...",
  "recomendacion_visual": "imagen|html|combinada",
  "visual_tipo": "integral|complex|function|geometry|default"
}
Reglas:
- Todo debe venir de Gemma 4: perfil, micro-retos, explicacion, ejemplos y recomendacion visual.
- Empieza desde lo mas accesible compatible con el perfil.
- Usa solo delimitadores LaTeX \\( \\) y \\[ \\].
- Los micro-retos deben ser claros, breves y con exactamente 4 opciones.
- La practica debe ser prioritaria.
'''

FEEDBACK_SYSTEM = r'''
Eres un editor de lecciones personalizadas. Recibes una leccion y feedback del usuario.
Devuelve SOLO el JSON completo actualizado. Manten coherencia con el perfil del estudiante.
'''

NOTES_SYSTEM = r'''
Eres un asistente que lee fotos de apuntes de estudio (manuscritos o impresos) y le explica al estudiante,
en tono humano y cercano, que encontraste en la imagen.

TU UNICA SALIDA debe ser un objeto JSON, sin texto antes ni despues, sin markdown, sin bloques de codigo.

Formato EXACTO de salida:
{
  "respuesta_natural": "...",
  "resumen": "...",
  "conceptos": ["..."],
  "dudas": ["..."],
  "texto_fuente": "..."
}

Reglas para cada campo:
- "respuesta_natural": de 2 a 5 frases en espanol, en primera persona, como si le hablaras directamente al
  estudiante (ej. "Vi tus apuntes sobre... Anotaste que... Te recomendaria revisar..."). Este texto se va a
  convertir en audio y a mostrarse tal cual en pantalla, asi que debe sonar natural al hablarlo: sin listas,
  sin simbolos, sin markdown.
- "resumen": resumen tecnico breve del contenido (texto de referencia, no necesita sonar hablado).
- "conceptos": lista de conceptos clave detectados en la imagen.
- "dudas": posibles dudas o puntos confusos que el estudiante podria tener sobre lo escrito.
- "texto_fuente": transcripcion lo mas fiel posible de lo que esta escrito en la imagen.

Si la imagen esta borrosa, incompleta, o no parece relacionada con apuntes de estudio, dilo honestamente en
"respuesta_natural" (ej. "No logro leer bien tus apuntes, ¿puedes tomar la foto con mejor luz o mas cerca?")
y llena los demas campos con lo poco que puedas rescatar, sin inventar contenido que no este en la imagen.
'''

INTERVIEW_QUESTIONS = [
    'Como te llamas o como quieres que te llame la app?',
    'Que materia o tema quieres dominar primero?',
    'Como describirias tu nivel actual en ese tema?',
    'Prefieres explicaciones intuitivas, formales, mixtas o con muchos ejemplos?',
    'Que formato te ayuda mas: audio, visual, practica, lectura o mezcla?',
    'Cuanto tiempo quieres estudiar por sesion?',
    'Que parte suele bloquearte mas o frustrarte?',
    'Para que quieres aprender esto en este momento?'
]


EXPERIENCE_SYSTEM = r'''
Eres Gemma, un disenador de experiencias educativas personalizadas y creativas. Recibes el perfil del
estudiante (incluye grupo_etario y necesidades_especiales), una instruccion en lenguaje natural sobre que
experiencia quiere el estudiante ahora mismo (ej. "escribeme un cuento sobre esto", "ayudame a leer este
tema paso a paso", "hazme preguntas de repaso"), y apuntes/temario de contexto.

IMPORTANTE: tu trabajo es UNICAMENTE redactar el contenido. NO generes HTML ni CSS: el sistema arma la
pagina automaticamente a partir de tu texto, ya adaptada al perfil (letras grandes y diseno simple si es
adulto_mayor o tiene necesidades_especiales, tono calido si es nino, diseno limpio en el resto). Esto hace
que tu respuesta sea mucho mas corta y rapida de generar.

Tu trabajo:
1. Decide que tipo de experiencia encaja mejor con la instruccion (ej: cuento, historia, lectura_guiada,
   preguntas_repaso, explicacion_simple, ejercicio_practico, resumen_narrado, u otro nombre corto).
2. Escribe el contenido textual completo, de forma CREATIVA y COMPLEMENTARIA, usando los apuntes/temario
   como fuente principal. Si el estudiante es nuevo en el tema, empieza con definiciones sencillas y luego
   profundiza. Menciona explicitamente que contexto estas usando (ej. "Basandome en los apuntes que
   subiste...").
3. Divide el contenido en parrafos cortos, separados por una linea en blanco (el sistema los convierte en
   secciones legibles automaticamente al armar el HTML). Usa entre 3 y 8 parrafos segun la profundidad
   necesaria: no te extiendas mas de lo necesario.
4. Si el contenido se presta para narrarse en voz alta (cuentos, historias, lecturas guiadas), escribe
   tambien un guion de audio breve y natural en espanol (3-6 frases). Si no aplica, escribe NINGUNO.

Responde EXCLUSIVAMENTE con este formato de bloques de texto plano (NO uses JSON, NO generes HTML):

===TIPO===
(tipo_experiencia en una sola linea)
===TITULO===
(titulo corto de la experiencia)
===TEXTO===
(contenido completo, en parrafos separados por linea en blanco, listo para leer)
===AUDIO===
(guion de audio breve, o la palabra NINGUNO si no aplica)
===FIN===

Reglas estrictas de formato: cada marcador va SOLO en su propia linea, sin texto extra en esa misma linea.
No escribas nada antes de ===TIPO=== ni despues de ===FIN===.
'''


## 6. Perfil del estudiante y temario desde PDF

In [6]:
_NOMBRE_PATTERNS = [
    r"\bme llamo\s+([A-ZÁÉÍÓÚÑ][a-zA-ZÁÉÍÓÚÑñ]+)",
    r"\bmi nombre es\s+([A-ZÁÉÍÓÚÑ][a-zA-ZÁÉÍÓÚÑñ]+)",
    r"\bllamame\s+([A-ZÁÉÍÓÚÑ][a-zA-ZÁÉÍÓÚÑñ]+)",
    r"\bsoy\s+([A-ZÁÉÍÓÚÑ][a-zA-ZÁÉÍÓÚÑñ]+)\b(?!\s+(?:un|una|el|la|de|estudiante|nuevo|nueva))",
]


def _regex_fallback_nombre(texto: str) -> str:
    """Red de seguridad, no un reemplazo de Gemma: solo se usa si Gemma dejo
    'nombre' vacio en su JSON a pesar de que el texto si menciona un nombre de
    forma explicita. Si Gemma ya devolvio algo, esta funcion nunca se llama."""
    for patron in _NOMBRE_PATTERNS:
        m = re.search(patron, texto, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip().capitalize()
    return ''


def build_profile_from_interview(interview_text: str) -> LearnerProfile:
    interview_text = (interview_text or '').strip()
    if not interview_text:
        raise ValueError('No hay texto de entrevista para construir el perfil.')

    raw = gemma_generate(
        PROFILE_SYSTEM,
        f'Entrevista transcrita del estudiante:\n{interview_text}',
        max_new_tokens=1800,
    )
    # required_keys=['nombre'] hace que quede loggeado en consola cuando Gemma
    # no lo llena, en vez de fallar en silencio via el default de Pydantic.
    data = extract_json_block(raw, required_keys=['nombre'], debug_label='perfil')

    if not str(data.get('nombre') or '').strip():
        rescatado = _regex_fallback_nombre(interview_text)
        data['nombre'] = rescatado or 'Estudiante'
        print(f"[build_profile_from_interview] Gemma no devolvio 'nombre'; se uso fallback regex -> '{data['nombre']}'")

    profile = LearnerProfile(**data)
    write_json(PROFILE_PATH, profile.model_dump())
    return profile


def load_profile() -> Optional[LearnerProfile]:
    if not PROFILE_PATH.exists():
        return None
    try:
        return LearnerProfile(**read_json(PROFILE_PATH, {}))
    except Exception:
        return None


def extract_pdf_content(pdf_path: str) -> List[PageBlock]:
    doc = fitz.open(pdf_path)
    blocks = []
    for page_idx, page in enumerate(doc):
        text = page.get_text('text')
        pix = page.get_pixmap(matrix=fitz.Matrix(1.4, 1.4))
        img_path = ASSETS / f'page_{page_idx + 1}.png'
        pix.save(str(img_path))
        blocks.append(PageBlock(page=page_idx + 1, text=text, image_paths=[str(img_path)]))
    doc.close()
    return blocks


def pdf_blocks_to_text(blocks: List[PageBlock], max_chars: int = 18000) -> str:
    return '\n\n'.join([f'PAGINA {b.page}\n{b.text}' for b in blocks])[:max_chars]


def build_curriculum_from_pdf_text(pdf_text: str) -> Materia:
    raw = gemma_generate(CURRICULUM_SYSTEM, f'Texto del temario:\n{pdf_text}', max_new_tokens=2600)
    return Materia(**extract_json_block(raw))

## 7. Progreso y favoritos

In [7]:
def mark_progress(subtopic: str, status: str):
    progress = read_json(PROGRESS_PATH, {})
    progress[subtopic] = {
        'status': status,
        'updated_at': datetime.datetime.now().isoformat(timespec='seconds')
    }
    write_json(PROGRESS_PATH, progress)


def get_progress() -> dict:
    return read_json(PROGRESS_PATH, {})


def save_favorite(title: str, kind: str, path: str, subtopic: str, reason: str = 'me gusto'):
    favs = read_json(FAVORITES_PATH, [])
    favs.append({
        'title': title,
        'kind': kind,
        'path': path,
        'subtopic': subtopic,
        'reason': reason,
        'timestamp': datetime.datetime.now().isoformat(timespec='seconds')
    })
    write_json(FAVORITES_PATH, favs)


def get_favorites() -> list:
    return read_json(FAVORITES_PATH, [])

## 8. Generación de lecciones y feedback

In [8]:
def generate_lesson(profile: LearnerProfile, tema_padre: str, subtema: Subtema, nivel: str = 'principiante', enfoque: str = '') -> Leccion:
    progress = read_json(PROGRESS_PATH, {})
    prompt = f'''Perfil del estudiante:
{json.dumps(profile.model_dump(), ensure_ascii=False, indent=2)}

Tema padre: {tema_padre}
Subtema: {subtema.nombre}
Tipo de concepto: {subtema.tipo_de_concepto}
Prerrequisitos: {", ".join(subtema.prerrequisitos) or "ninguno"}
Errores comunes: {", ".join(subtema.errores_comunes) or "ninguno"}
Nivel tecnico solicitado: {nivel}
Enfoque adicional: {enfoque or "sin enfoque adicional"}
Estado de progreso actual: {json.dumps(progress.get(subtema.nombre, {}), ensure_ascii=False)}'''
    raw = gemma_generate(LESSON_SYSTEM, prompt, max_new_tokens=3600)
    data = extract_json_block(raw)
    lesson = Leccion(
        titulo=data['titulo'],
        tema_padre=tema_padre,
        subtema=subtema.nombre,
        nivel_tecnico=nivel,
        enfoque_usuario=enfoque,
        conceptos_previos=data.get('conceptos_previos', []),
        mapa_conocimiento=data.get('mapa_conocimiento', []),
        explicacion_paso_a_paso=data.get('explicacion_paso_a_paso', []),
        pausa_reflexion=data.get('pausa_reflexion', ''),
        glosario=data.get('glosario', {}),
        micro_retos=[MicroReto(**r) for r in data.get('micro_retos', [])],
        ejemplos=[EjemploNivel(**e) for e in data.get('ejemplos', [])],
        historia_base=data.get('historia_base', ''),
        guion_audio=data.get('guion_audio', ''),
        latex_clave=data.get('latex_clave', []),
        prompt_ilustracion=data.get('prompt_ilustracion', ''),
        html_prompt=data.get('html_prompt', ''),
        recomendacion_visual=data.get('recomendacion_visual', 'html'),
        visual_tipo=data.get('visual_tipo', 'default'),
    )
    memory = read_json(LESSONS_PATH, {})
    memory[subtema.nombre] = lesson.model_dump()
    write_json(LESSONS_PATH, memory)
    mark_progress(subtema.nombre, 'started')
    return lesson


def apply_feedback(lesson: Leccion, comment: str) -> Leccion:
    payload = {'lesson': lesson.model_dump(), 'feedback': comment}
    raw = gemma_generate(FEEDBACK_SYSTEM, json.dumps(payload, ensure_ascii=False), max_new_tokens=3200)
    data = extract_json_block(raw)
    updated = Leccion(**{**lesson.model_dump(), **data})
    updated.version = lesson.version + 1
    updated.feedback_historial = lesson.feedback_historial + [comment]
    memory = read_json(LESSONS_PATH, {})
    memory[lesson.subtema] = updated.model_dump()
    write_json(LESSONS_PATH, memory)
    return updated


def load_lesson(subtema: str) -> Optional[Leccion]:
    memory = read_json(LESSONS_PATH, {})
    data = memory.get(subtema)
    return Leccion(**data) if data else None

## 9. Lectura de apuntes por imagen (multimodal)

In [9]:
def extract_notes_from_images(image_paths: List[str]) -> dict:
    default = {
        'respuesta_natural': '',
        'resumen': '',
        'conceptos': [],
        'dudas': [],
        'texto_fuente': '',
    }
    if not image_paths:
        return default

    user_prompt = (
        'Lee estas fotos de apuntes y explicale al estudiante, en tono natural y hablado, que encontraste. '
        'Si hay formulas, terminos o fragmentos incompletos, interpretalos con cautela.'
    )
    raw = gemma_multimodal_generate(NOTES_SYSTEM, user_prompt, image_paths, max_new_tokens=2200)

    try:
        data = extract_json_block(raw, required_keys=['respuesta_natural', 'texto_fuente'], debug_label='notas_imagen')
    except ValueError:
        # Gemma no devolvio JSON valido: no tiramos el trabajo del modelo, usamos su
        # texto crudo como respuesta natural para que el usuario reciba algo util
        # de todas formas, y lo dejamos loggeado para poder depurar el prompt.
        print('[extract_notes_from_images] Gemma no devolvio JSON valido, usando texto crudo como respaldo.')
        texto_crudo = raw.strip()
        data = {'respuesta_natural': texto_crudo[:800], 'resumen': texto_crudo[:400]}

    merged = {**default, **data}
    write_json(NOTE_IMAGES_PATH, {'images': image_paths, 'analysis': merged})
    return merged


def enrich_lesson_with_notes(lesson: Leccion, notes_data: dict) -> Leccion:
    if not notes_data or not notes_data.get('texto_fuente'):
        return lesson
    payload = {
        'lesson': lesson.model_dump(),
        'notes': notes_data,
        'instruction': 'Reescribe y enriquece la leccion usando los apuntes del estudiante como contexto prioritario. Manten estructura didactica y micro-retos.'
    }
    raw = gemma_generate(FEEDBACK_SYSTEM, json.dumps(payload, ensure_ascii=False), max_new_tokens=3200)
    data = extract_json_block(raw)
    updated = Leccion(**{**lesson.model_dump(), **data})
    updated.version = lesson.version + 1
    updated.feedback_historial = lesson.feedback_historial + ['Enriquecida con apuntes por imagen']
    memory = read_json(LESSONS_PATH, {})
    memory[lesson.subtema] = updated.model_dump()
    write_json(LESSONS_PATH, memory)
    return updated

## 10. Artefactos: audio (TTS), imagen de vista previa, HTML interactivo y visualización Plotly

In [10]:
def _slug(text: str) -> str:
    return re.sub(r'[^a-zA-Z0-9]+', '_', text)[:40]


def generate_audio(lesson: Leccion) -> str:
    out = MEDIA / f'audio_{_slug(lesson.titulo)}_v{lesson.version}.mp3'
    gTTS(text=lesson.guion_audio or lesson.historia_base or lesson.titulo, lang='es', slow=False).save(str(out))
    return str(out)


def text_to_audio_path(text: str, name_hint: str = 'clip') -> Optional[str]:
    text = (text or '').strip()
    if not text:
        return None
    out = MEDIA / f'tts_{_slug(name_hint)}_{uuid.uuid4().hex[:8]}.mp3'
    gTTS(text=text, lang='es', slow=False).save(str(out))
    return str(out)


def generate_preview_image(lesson: Leccion) -> str:
    img = Image.new('RGB', (1400, 900), color=(248, 250, 252))
    draw = ImageDraw.Draw(img)
    draw.text((50, 50), lesson.titulo, fill=(17, 35, 62))
    draw.text((50, 120), f'Nivel: {lesson.nivel_tecnico}', fill=(40, 70, 100))
    draw.text((50, 180), textwrap.fill('Conceptos previos: ' + ', '.join(lesson.conceptos_previos[:6]), width=70), fill=(65, 65, 80))
    body = '\n\n'.join([f'{i + 1}. {s}' for i, s in enumerate(lesson.explicacion_paso_a_paso[:5])])
    draw.text((50, 300), textwrap.fill(body, width=72), fill=(20, 20, 20))
    out = ASSETS / f'preview_{_slug(lesson.titulo)}_v{lesson.version}.png'
    img.save(out)
    return str(out)


def build_concept_visual(lesson: Leccion) -> str:
    slug = _slug(lesson.titulo)
    out = HTML / f'visual_{slug}_v{lesson.version}.html'
    vt = lesson.visual_tipo
    if vt == 'integral':
        x = [i / 50 for i in range(0, 151)]
        y = [v ** 0.7 + 0.8 for v in x]
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color='#2563eb', width=4), name='f(x)'))
        fig.add_trace(go.Scatter(x=[0.5, 0.5, 2.5, 2.5, 0.5], y=[0, 1.1, 1.1, 0, 0], fill='toself', fillcolor='rgba(59,130,246,0.2)', line=dict(color='rgba(59,130,246,0.0)'), name='Area'))
        fig.update_layout(title=lesson.titulo, xaxis_title='x', yaxis_title='y', height=520, template='plotly_white')
    elif vt == 'complex':
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0, 2, 3], y=[0, 1, 3], mode='lines+markers+text', text=['0', '2+i', '3+3i'], textposition='top center', line=dict(color='#0f766e', width=4)))
        fig.update_layout(title=lesson.titulo, xaxis_title='Re', yaxis_title='Im', height=520, template='plotly_white', xaxis=dict(scaleanchor='y', scaleratio=1))
    elif vt == 'geometry':
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0, 2, 4, 0], y=[0, 3, 0, 0], mode='lines+markers', fill='toself', line=dict(color='#db2777', width=4)))
        fig.update_layout(title=lesson.titulo, height=520, template='plotly_white')
    else:
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0, 1, 2, 3], y=[1, 3, 2, 4], mode='lines+markers', line=dict(color='#7c3aed', width=4)))
        fig.update_layout(title=lesson.titulo, height=520, template='plotly_white')
    fig.write_html(str(out), include_plotlyjs='cdn', full_html=True, config={'displaylogo': False, 'responsive': True})
    return str(out)


def generate_practice_html(lesson: Leccion) -> str:
    out = HTML / f'{_slug(lesson.titulo)}_v{lesson.version}.html'
    glossary_html = ''.join([f'<li><strong>{k}</strong>: {v}</li>' for k, v in lesson.glosario.items()])
    retos_html = []
    for j, r in enumerate(lesson.micro_retos, start=1):
        opts = ''.join([f'<button class="opt" data-correct="{1 if idx == r.respuesta_correcta else 0}">{chr(65 + idx)}. {op}</button>' for idx, op in enumerate(r.opciones)])
        retos_html.append(f'''
        <div class="mcq" data-q="{j}">
          <p class="q"><strong>Reto {j}:</strong> {r.pregunta}</p>
          <div class="opts">{opts}</div>
          <details><summary>Ver verificacion</summary><p class="mcq-exp">{r.explicacion}</p></details>
        </div>
        ''')
    examples_html = []
    for i, ex in enumerate(lesson.ejemplos, start=1):
        examples_html.append(f'''
        <details class="example-card">
          <summary>Ejemplo {i} - {ex.nivel.title()}</summary>
          <p><strong>Enunciado:</strong> {ex.enunciado}</p>
          <p><strong>Pista:</strong> {ex.pista}</p>
          <details>
            <summary>Ver respuesta</summary>
            <div class="answer">{ex.respuesta}</div>
          </details>
        </details>
        ''')
    html = f'''<!DOCTYPE html>
<html lang="es"><head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>{lesson.titulo}</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/contrib/auto-render.min.js"
onload="renderMathInElement(document.body, {{delimiters:[{{left:'\\\\(', right:'\\\\)', display:false}},{{left:'\\\\[', right:'\\\\]', display:true}}]}});"></script>
<style>
body{{font-family:Inter,Arial,sans-serif;background:#f5f7fb;color:#17212b;margin:0;padding:32px;line-height:1.6;}}
.wrap{{max-width:1100px;margin:0 auto;background:#fff;padding:32px;border-radius:20px;box-shadow:0 10px 40px rgba(0,0,0,.08);}}
.card{{background:#f8fafc;border:1px solid #dde5f0;border-radius:16px;padding:20px;margin-top:20px;}}
.example-card,.mcq{{margin:16px 0;padding:12px 16px;background:#f9fbff;border:1px solid #d8e3f4;border-radius:14px;}}
.answer{{margin-top:10px;padding:12px;background:#eef6ea;border-radius:10px;}}
.pause{{background:#fff6db;border-left:5px solid #dfad11;padding:14px 18px;border-radius:10px;}}
.opts{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;}}
.opt{{padding:12px;border-radius:12px;border:1px solid #cdd8e7;background:white;cursor:pointer;text-align:left;}}
.opt.selected{{outline:3px solid #2b6cb0;}}
.verdict{{margin-top:12px;padding:12px;border-radius:12px;display:none;}}
.verdict.ok{{background:#e7f7e9;border:1px solid #88c48e;display:block;}}
.verdict.no{{background:#fdeaea;border:1px solid #e08b8b;display:block;}}
</style></head>
<body><div class="wrap">
<h1>{lesson.titulo}</h1>
<p><strong>Subtema:</strong> {lesson.subtema} - <strong>Nivel:</strong> {lesson.nivel_tecnico}</p>
<section class="card"><h2>Conceptos previos</h2><ul>{''.join([f'<li>{c}</li>' for c in lesson.conceptos_previos])}</ul></section>
<section class="card"><h2>Mapa de conocimiento</h2><ol>{''.join([f'<li>{m}</li>' for m in lesson.mapa_conocimiento])}</ol></section>
<section class="card"><h2>Explicacion paso a paso</h2><ol>{''.join([f'<li>{s}</li>' for s in lesson.explicacion_paso_a_paso])}</ol><div class="pause"><strong>Toma una pausa:</strong> {lesson.pausa_reflexion}</div></section>
<section class="card"><h2>Glosario inicial</h2><ul>{glossary_html}</ul></section>
<section class="card"><h2>Micro-retos</h2>{''.join(retos_html)}</section>
<section class="card"><h2>Practica por niveles</h2>{''.join(examples_html)}</section>
</div>
<script>
document.querySelectorAll('.mcq').forEach(card => {{
  const buttons = [...card.querySelectorAll('.opt')];
  const summary = document.createElement('div');
  summary.className = 'verdict';
  card.appendChild(summary);
  buttons.forEach(btn => btn.addEventListener('click', () => {{
    buttons.forEach(b => b.classList.remove('selected'));
    btn.classList.add('selected');
    const correct = btn.dataset.correct === '1';
    summary.textContent = correct ? 'Correcto. Ahora abre \u201cVer verificacion\u201d para revisar la explicacion.' : 'Respuesta registrada. Usa \u201cVer verificacion\u201d para comprobarla.';
    summary.className = 'verdict ' + (correct ? 'ok' : 'no');
  }}));
}});
</script>
</body></html>'''
    out.write_text(html, encoding='utf-8')
    return str(out)


def export_generated_text(subtema: str, content: str, formato: str = 'txt') -> str:
    ext = 'tex' if formato == 'latex' else 'txt'
    out = EXPORT_DIR / f'{_slug(subtema)}_{uuid.uuid4().hex[:8]}.{ext}'
    out.write_text(content, encoding='utf-8')
    return str(out)

## 11. Estado en memoria del servidor

Guarda el perfil activo y el temario cargado durante la sesión del servidor (equivalente al `STATE` que antes usaba la UI de Gradio).

In [11]:
STATE: Dict[str, Any] = {
    'profile': load_profile(),
    'materia': None,
    'flat_topics': [],   # lista de (tema_padre, Subtema)
}


def get_flat_topic(index: int):
    topics = STATE.get('flat_topics', [])
    if index < 0 or index >= len(topics):
        raise IndexError('topic_index fuera de rango. Consulta GET /curriculum/topics')
    return topics[index]

## 11 bis. Generacion de experiencias personalizadas (Gemma escribe su propio HTML)

Nueva capacidad: a partir de una instruccion en texto o en audio (ej. "escribeme un cuento sobre esto", "ayudame a leer este tema"), Gemma 4 decide el tipo de experiencia, redacta el contenido usando los apuntes/temario ya cargados como fuente, y genera su propia pagina HTML autocontenida. Si el perfil activo es de un adulto mayor (o pide algo simple), Gemma recibe instrucciones explicitas para producir una interfaz muy sencilla, con letras grandes y minima navegacion.

No se usa JSON para este contenido (el HTML generado rompe facilmente el formato JSON), sino un formato de bloques delimitados que se parsea con una funcion propia, con un HTML de respaldo por si Gemma no sigue el formato al pie de la letra.


In [12]:
EXPERIENCES_PATH = DATA / 'experiences.json'
if not EXPERIENCES_PATH.exists():
    write_json(EXPERIENCES_PATH, [])


def _collect_apuntes_context(subtema: Optional[str] = None, max_chars: int = 6000) -> str:
    partes = []
    materia = STATE.get('materia')
    if materia:
        partes.append(f'Materia: {materia.materia} - Unidad: {materia.unidad}')
        for tema in materia.temas:
            nombres = ', '.join([s.nombre for s in tema.subtemas])
            partes.append(f'Tema: {tema.tema} -> Subtemas: {nombres}')
    notas = read_json(NOTE_IMAGES_PATH, {})
    analysis = notas.get('analysis') if isinstance(notas, dict) else None
    if analysis:
        if analysis.get('resumen'):
            partes.append(f"Resumen de apuntes en imagen: {analysis['resumen']}")
        if analysis.get('texto_fuente'):
            partes.append(f"Texto de apuntes: {analysis['texto_fuente']}")
    if subtema:
        lesson = load_lesson(subtema)
        if lesson:
            partes.append(f'Leccion existente sobre "{subtema}": ' + '\n'.join(lesson.explicacion_paso_a_paso))
    texto = '\n\n'.join(partes)
    return texto[:max_chars] if texto else 'No hay apuntes ni temario cargados todavia.'


def _parse_experience_blocks(raw: str) -> dict:
    """Parsea el formato de bloques ===MARCADOR=== que escribe Gemma.

    OPTIMIZACION: ya no hay bloque ===HTML=== que parsear (Gemma dejo de generar
    HTML, ver EXPERIENCE_SYSTEM). Esto simplifica el parseo y elimina el bug
    original de HTML cortado, porque ya no hay HTML que se pueda cortar.
    """
    raw = raw.strip()

    def _grab(marker_ini, marker_fin, text):
        start = text.find(marker_ini)
        if start == -1:
            return ''
        start += len(marker_ini)
        end = text.find(marker_fin, start)
        chunk = text[start:end] if end != -1 else text[start:]
        return chunk.strip()

    tipo = _grab('===TIPO===', '===TITULO===', raw) or 'experiencia'
    titulo = _grab('===TITULO===', '===TEXTO===', raw) or 'Experiencia personalizada'
    texto = _grab('===TEXTO===', '===AUDIO===', raw)
    audio_guion = _grab('===AUDIO===', '===FIN===', raw)

    if not texto:
        # Respaldo: si Gemma no siguio el formato de marcadores al pie de la
        # letra, igual mostramos su salida cruda en vez de dejar la experiencia vacia.
        print('[_parse_experience_blocks] No se encontro ===TEXTO===, usando salida cruda como respaldo.')
        texto = raw

    if audio_guion.strip().upper() == 'NINGUNO':
        audio_guion = ''

    return {
        'tipo_experiencia': tipo.strip(),
        'titulo': titulo.strip(),
        'contenido_texto': texto.strip(),
        'guion_audio': audio_guion.strip(),
    }


def _render_experience_html(tipo: str, titulo: str, texto: str, profile: LearnerProfile) -> str:
    """Arma el HTML en Python (instantaneo, sin gastar tokens de Gemma en escribir
    markup) respetando el perfil: letras grandes y diseno simple para adulto_mayor
    o necesidades_especiales, tono calido para nino, diseno limpio en el resto.
    Antes esta logica se le pedia a Gemma en texto libre; ahora es codigo, asi
    que es consistente en cada llamada y no consume tokens de generacion."""
    parrafos = [p.strip() for p in texto.split('\n\n') if p.strip()]
    cuerpo = '\n'.join(f'<p>{p}</p>' for p in parrafos) or f'<p>{texto}</p>'

    simple = profile.grupo_etario == 'adulto_mayor' or bool((profile.necesidades_especiales or '').strip())
    ninos = profile.grupo_etario == 'nino'

    if simple:
        font_size, titulo_size, bg, accent, max_w = '24px', '38px', '#ffffff', '#1d4ed8', '760px'
    elif ninos:
        font_size, titulo_size, bg, accent, max_w = '19px', '30px', '#fff7ed', '#ea580c', '820px'
    else:
        font_size, titulo_size, bg, accent, max_w = '17px', '28px', '#f5f7fb', '#2563eb', '900px'

    return f'''<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>{titulo}</title>
<style>
body{{font-family:Arial,sans-serif;background:{bg};color:#17212b;padding:32px;line-height:1.7;font-size:{font_size};}}
.wrap{{max-width:{max_w};margin:0 auto;background:#fff;padding:32px;border-radius:16px;box-shadow:0 2px 12px rgba(0,0,0,0.06);}}
h1{{font-size:{titulo_size};color:{accent};margin-top:0;}}
.tag{{display:inline-block;background:{accent};color:#fff;padding:4px 12px;border-radius:999px;font-size:14px;margin-bottom:16px;}}
p{{margin:0 0 18px 0;}}
</style></head><body><div class="wrap">
<span class="tag">{tipo}</span>
<h1>{titulo}</h1>
{cuerpo}
</div></body></html>'''


def generate_experience(profile: LearnerProfile, instruccion: str, contexto_extra: str = '', subtema: Optional[str] = None) -> dict:
    apuntes = _collect_apuntes_context(subtema)
    prompt = f'''Perfil del estudiante:
{json.dumps(profile.model_dump(), ensure_ascii=False, indent=2)}

Instruccion del estudiante (que experiencia quiere ahora mismo):
{instruccion}

Contexto adicional dado por el estudiante:
{contexto_extra or "sin contexto adicional"}

Apuntes / temario disponible como fuente de contenido:
{apuntes}'''

    # OPTIMIZACION CLAVE: antes max_new_tokens=6000 porque Gemma tambien tenia que
    # escribir un HTML completo autocontenido (facilmente >4000 tokens el HTML solo).
    # Ahora Gemma SOLO redacta contenido (el HTML lo arma _render_experience_html en
    # Python), asi que 1400 tokens sobran de sobra y la generacion es varias veces
    # mas rapida y ya no se corta a la mitad.
    raw = gemma_generate(EXPERIENCE_SYSTEM, prompt, max_new_tokens=1400, temperature=0.45)
    parsed = _parse_experience_blocks(raw)
    if not parsed['contenido_texto']:
        print('[generate_experience] Advertencia: Gemma no genero contenido_texto para esta experiencia.')

    html = _render_experience_html(parsed['tipo_experiencia'], parsed['titulo'], parsed['contenido_texto'], profile)

    slug = _slug(parsed['titulo'] or instruccion)
    html_path = HTML / f'experiencia_{slug}_{uuid.uuid4().hex[:8]}.html'
    html_path.write_text(html, encoding='utf-8')

    audio_path = None
    if parsed['guion_audio']:
        audio_path = text_to_audio_path(parsed['guion_audio'], name_hint=f'experiencia_{slug}')

    registro = {
        'id': uuid.uuid4().hex[:10],
        'tipo_experiencia': parsed['tipo_experiencia'],
        'titulo': parsed['titulo'],
        'instruccion': instruccion,
        'subtema': subtema or '',
        'contenido_texto': parsed['contenido_texto'],
        'html_path': str(html_path),
        'audio_path': audio_path,
        'grupo_etario_usado': profile.grupo_etario,
        'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
    }
    historial = read_json(EXPERIENCES_PATH, [])
    historial.append(registro)
    write_json(EXPERIENCES_PATH, historial)
    return registro


## 12. Servidor FastAPI — todos los endpoints

| Método | Ruta | Qué hace |
|---|---|---|
| GET | `/health` | Estado del servicio y del modelo |
| POST | `/profile/interview` | Crea el perfil a partir de texto de entrevista |
| POST | `/profile/audio` | Crea el perfil a partir de un audio (se transcribe con Whisper) |
| GET | `/profile` | Perfil activo |
| GET | `/profile/preguntas` | Preguntas sugeridas de entrevista |
| POST | `/curriculum/pdf` | Sube un PDF y genera el temario estructurado |
| GET | `/curriculum/topics` | Lista de temas/subtemas detectados, con su índice |
| POST | `/lesson` | Genera una lección para `topic_index` |
| GET | `/lesson/{subtema}` | Recupera una lección ya generada |
| POST | `/lesson/{subtema}/feedback` | Regenera la lección con retroalimentación |
| POST | `/lesson/{subtema}/enrich-with-notes` | Enriquece la lección con los apuntes de fotos ya analizados |
| GET | `/lesson/{subtema}/audio` | Descarga el audio (mp3) de la lección |
| GET | `/lesson/{subtema}/html` | Descarga el HTML interactivo con micro-retos |
| GET | `/lesson/{subtema}/visual` | Descarga la visualización Plotly en HTML |
| GET | `/lesson/{subtema}/preview-image` | Descarga la imagen de vista previa (PNG) |
| POST | `/lesson/{subtema}/quiz/verify` | Verifica una respuesta del micro-reto final |
| POST | `/lesson/{subtema}/repaso` | Genera preguntas de repaso con Gemma |
| POST | `/notes/images` | Sube fotos de apuntes/libro y las analiza con Gemma multimodal |
| POST | `/progress` | Marca el estado de un subtema |
| GET | `/progress` | Progreso completo |
| POST | `/favorites` | Marca una lección como favorita |
| GET | `/favorites` | Lista de favoritos |
| POST | `/export` | Exporta contenido a `.txt` o `.tex` descargable |
| POST | `/audio/transcribe` | Transcribe cualquier audio con Whisper |
| POST | `/tts` | Convierte texto a audio (mp3) |
| POST | `/ask` | Pregunta libre a Gemma 4 |
| POST | `/experience` | Genera una experiencia personalizada (cuento, lectura, preguntas, etc.) a partir de texto |
| POST | `/experience/audio` | Igual que `/experience` pero recibiendo audio (se transcribe con Whisper) |
| GET | `/experience/historial` | Lista todas las experiencias generadas |
| GET | `/experience/{experience_id}` | Recupera una experiencia generada por id |

También se montan `/files/media`, `/files/html`, `/files/assets` y `/files/exports` como carpetas estáticas, por si prefieres acceder a los archivos generados directamente por URL en lugar de por los endpoints de descarga.

In [13]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel as _BM
from typing import List as _List, Optional as _Opt

app = FastAPI(title='Gemma 4 Study Companion API', version='1.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Sirve los archivos generados directamente por URL (audio, html, imagenes)
app.mount('/files/media', StaticFiles(directory=str(MEDIA)), name='media')
app.mount('/files/html', StaticFiles(directory=str(HTML)), name='html')
app.mount('/files/assets', StaticFiles(directory=str(ASSETS)), name='assets')
app.mount('/files/exports', StaticFiles(directory=str(EXPORT_DIR)), name='exports')


def _save_upload(upload: UploadFile, folder: Path) -> str:
    ext = Path(upload.filename or '').suffix or ''
    dest = folder / f'{uuid.uuid4().hex[:10]}{ext}'
    with open(dest, 'wb') as f:
        f.write(upload.file.read())
    return str(dest)


# ---------- Esquemas de request ----------
class InterviewBody(_BM):
    texto: str


class TopicBody(_BM):
    topic_index: int
    nivel: Literal['principiante', 'intermedio', 'avanzado'] = 'principiante'
    enfoque: str = ''


class FeedbackBody(_BM):
    comentario: str


class QuizBody(_BM):
    respuesta_index: int


class ProgressBody(_BM):
    subtema: str
    estado: str


class FavoriteBody(_BM):
    subtema: str
    motivo: str = 'me gusto'


class ExportBody(_BM):
    subtema: str
    formato: Literal['txt', 'latex'] = 'txt'
    contenido: _Opt[str] = None


class AskBody(_BM):
    pregunta: str
    contexto: str = ''


# ---------- Salud ----------
@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID, 'gpu': torch.cuda.is_available()}


# ---------- Perfil ----------
@app.post('/profile/interview')
def profile_interview(body: InterviewBody):
    try:
        profile = build_profile_from_interview(body.texto)
        STATE['profile'] = profile
        return profile.model_dump()
    except Exception as e:
        raise HTTPException(400, f'No se pudo construir el perfil: {e}')


@app.post('/profile/audio')
async def profile_audio(audio: UploadFile = File(...), texto_extra: str = Form('')):
    path = _save_upload(audio, UPLOADS)
    transcript = transcribe_audio_safe(path)
    full_text = (transcript + '\n' if transcript else '') + (texto_extra or '')
    if not full_text.strip():
        raise HTTPException(400, 'No se pudo transcribir audio ni se recibio texto de apoyo.')
    try:
        profile = build_profile_from_interview(full_text)
        STATE['profile'] = profile
        return {'perfil': profile.model_dump(), 'transcripcion': transcript}
    except Exception as e:
        raise HTTPException(400, f'No se pudo construir el perfil: {e}')


@app.get('/profile')
def get_profile():
    profile = STATE.get('profile') or load_profile()
    if not profile:
        raise HTTPException(404, 'Aun no hay perfil creado.')
    return profile.model_dump()


@app.get('/profile/preguntas')
def get_interview_questions():
    return {'preguntas': INTERVIEW_QUESTIONS}


# ---------- Temario / PDF ----------
@app.post('/curriculum/pdf')
async def curriculum_pdf(pdf: UploadFile = File(...)):
    path = _save_upload(pdf, UPLOADS)
    try:
        blocks = extract_pdf_content(path)
        text = pdf_blocks_to_text(blocks)
        materia = build_curriculum_from_pdf_text(text)
    except Exception as e:
        raise HTTPException(400, f'No se pudo procesar el PDF: {e}')
    flat = []
    for tema in materia.temas:
        for sub in tema.subtemas:
            flat.append((tema.tema, sub))
    STATE['materia'] = materia
    STATE['flat_topics'] = flat
    topics = [{'index': i, 'tema': t, 'subtema': s.nombre} for i, (t, s) in enumerate(flat)]
    return {'materia': materia.model_dump(), 'topics': topics}


@app.get('/curriculum/topics')
def curriculum_topics():
    flat = STATE.get('flat_topics', [])
    return {'topics': [{'index': i, 'tema': t, 'subtema': s.nombre} for i, (t, s) in enumerate(flat)]}


# ---------- Lecciones ----------
@app.post('/lesson')
def create_lesson(body: TopicBody):
    profile = STATE.get('profile') or load_profile()
    if not profile:
        raise HTTPException(400, 'Primero crea el perfil del estudiante en /profile/interview o /profile/audio.')
    try:
        tema, sub = get_flat_topic(body.topic_index)
    except IndexError as e:
        raise HTTPException(400, str(e))
    try:
        lesson = generate_lesson(profile, tema, sub, nivel=body.nivel, enfoque=body.enfoque)
        return lesson.model_dump()
    except Exception as e:
        raise HTTPException(500, f'Fallo generando la leccion: {e}')


@app.get('/lesson/{subtema}')
def get_lesson(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'No existe una leccion generada para ese subtema.')
    return lesson.model_dump()


@app.post('/lesson/{subtema}/feedback')
def lesson_feedback(subtema: str, body: FeedbackBody):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    try:
        updated = apply_feedback(lesson, body.comentario)
        return updated.model_dump()
    except Exception as e:
        raise HTTPException(500, f'Fallo aplicando feedback: {e}')


@app.post('/lesson/{subtema}/enrich-with-notes')
def lesson_enrich(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    note_data = read_json(NOTE_IMAGES_PATH, {})
    analysis = note_data.get('analysis') if isinstance(note_data, dict) else None
    if not analysis:
        raise HTTPException(400, 'No hay apuntes cargados. Usa /notes/images primero.')
    updated = enrich_lesson_with_notes(lesson, analysis)
    return updated.model_dump()


@app.get('/lesson/{subtema}/audio')
def lesson_audio(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    path = generate_audio(lesson)
    return FileResponse(path, media_type='audio/mpeg', filename=Path(path).name)


@app.get('/lesson/{subtema}/html')
def lesson_html(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    path = generate_practice_html(lesson)
    return FileResponse(path, media_type='text/html', filename=Path(path).name)


@app.get('/lesson/{subtema}/visual')
def lesson_visual(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    path = build_concept_visual(lesson)
    return FileResponse(path, media_type='text/html', filename=Path(path).name)


@app.get('/lesson/{subtema}/preview-image')
def lesson_preview_image(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    path = generate_preview_image(lesson)
    return FileResponse(path, media_type='image/png', filename=Path(path).name)


@app.post('/lesson/{subtema}/quiz/verify')
def quiz_verify(subtema: str, body: QuizBody):
    lesson = load_lesson(subtema)
    if not lesson or not lesson.micro_retos:
        raise HTTPException(404, 'No hay micro-retos disponibles para ese subtema.')
    quiz = lesson.micro_retos[-1]
    correcto = body.respuesta_index == quiz.respuesta_correcta
    return {'correcto': correcto, 'explicacion': quiz.explicacion}


@app.post('/lesson/{subtema}/repaso')
def lesson_repaso(subtema: str):
    lesson = load_lesson(subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion.')
    prompt = f'Genera preguntas breves de repaso en espanol para el subtema "{subtema}", basadas en esta leccion:\n{json.dumps(lesson.model_dump(), ensure_ascii=False)}'
    texto = gemma_generate('Eres un tutor que redacta preguntas de repaso claras y breves.', prompt, max_new_tokens=900)
    return {'preguntas_repaso': texto}


# ---------- Apuntes por imagen ----------
@app.post('/notes/images')
async def notes_images(imagenes: _List[UploadFile] = File(...)):
    paths = [_save_upload(img, UPLOADS) for img in imagenes]
    try:
        analysis = extract_notes_from_images(paths)
    except Exception as e:
        raise HTTPException(500, f'Fallo leyendo las imagenes: {e}')

    # El audio se genera a partir de lo que Gemma realmente redacto para el estudiante
    # (respuesta_natural), no de una plantilla fija tipo 'Se cargo la foto. '.
    texto_hablado = (analysis.get('respuesta_natural') or analysis.get('resumen') or 'Ya revise tus apuntes.').strip()
    audio_path = text_to_audio_path(texto_hablado, name_hint='notas')

    return {
        'analisis': analysis,
        'respuesta_natural': analysis.get('respuesta_natural', ''),
        'audio_confirmacion': audio_path,
    }


# ---------- Progreso ----------
@app.post('/progress')
def set_progress(body: ProgressBody):
    mark_progress(body.subtema, body.estado)
    return {'ok': True}


@app.get('/progress')
def all_progress():
    return get_progress()


# ---------- Favoritos ----------
@app.post('/favorites')
def add_favorite(body: FavoriteBody):
    lesson = load_lesson(body.subtema)
    if not lesson:
        raise HTTPException(404, 'Genera primero la leccion antes de marcarla como favorita.')
    html_path = str(HTML / f'{_slug(lesson.titulo)}_v{lesson.version}.html')
    save_favorite(lesson.titulo, 'lesson', html_path, lesson.subtema, body.motivo)
    mark_progress(lesson.subtema, 'liked')
    return {'ok': True}


@app.get('/favorites')
def list_favorites():
    return {'favoritos': get_favorites()}


# ---------- Exportar TXT / LaTeX ----------
@app.post('/export')
def export_text(body: ExportBody):
    lesson = load_lesson(body.subtema)
    content = body.contenido
    if not content:
        if not lesson:
            raise HTTPException(404, 'No hay leccion ni contenido explicito para exportar.')
        content = '\n\n'.join(lesson.explicacion_paso_a_paso) or lesson.historia_base
    path = export_generated_text(body.subtema, content, body.formato)
    return FileResponse(path, media_type='text/plain', filename=Path(path).name)


# ---------- Utilidades libres ----------
@app.post('/audio/transcribe')
async def audio_transcribe(audio: UploadFile = File(...)):
    path = _save_upload(audio, UPLOADS)
    return {'texto': transcribe_audio_safe(path)}


@app.post('/tts')
def tts(texto: str = Form(...)):
    path = text_to_audio_path(texto, name_hint='tts')
    if not path:
        raise HTTPException(400, 'Texto vacio.')
    return FileResponse(path, media_type='audio/mpeg', filename=Path(path).name)


@app.post('/ask')
def ask(body: AskBody):
    system = 'Eres un tutor personal que responde de forma clara, breve y en espanol.'
    prompt = body.pregunta if not body.contexto else f'Contexto: {body.contexto}\n\nPregunta: {body.pregunta}'
    respuesta = gemma_generate(system, prompt, max_new_tokens=1200)
    return {'respuesta': respuesta}

### Nuevos endpoints: perfil mas flexible + experiencias personalizadas

No se toco ningun endpoint existente. Se agregan `/experience`, `/experience/audio`, `/experience/historial` y `/experience/{experience_id}`.


In [14]:
class ExperienceBody(_BM):
    instruccion: str
    contexto_extra: str = ''
    subtema: _Opt[str] = None


def _experience_response(registro: dict, transcripcion: _Opt[str] = None) -> dict:
    resp = {
        'id': registro['id'],
        'tipo_experiencia': registro['tipo_experiencia'],
        'titulo': registro['titulo'],
        'subtema': registro['subtema'],
        'contenido_texto': registro['contenido_texto'],
        'url_html': f"/files/html/{Path(registro['html_path']).name}",
        'url_audio': f"/files/media/{Path(registro['audio_path']).name}" if registro['audio_path'] else None,
        'grupo_etario_usado': registro['grupo_etario_usado'],
        'timestamp': registro['timestamp'],
    }
    if transcripcion is not None:
        resp['transcripcion'] = transcripcion
    return resp


# OPTIMIZACION CLAVE (arregla el "ngrok se muere"): generar una experiencia toma
# varios segundos de GPU aunque ya se recorto el max_new_tokens. Si el endpoint
# se queda esperando (bloqueado) todo ese tiempo antes de responder, el tunel de
# ngrok / el cliente HTTP puede cortar la conexion por inactividad o timeout, y
# el resultado se pierde aunque el modelo si haya terminado de generar.
#
# Con este patron, el endpoint responde CASI INSTANTANEO con un job_id, la
# generacion corre en un hilo en background, y el cliente pregunta el resultado
# con GET /experience/status/{job_id} cada 2-3 segundos hasta que status='done'.
# Asi ninguna conexion HTTP queda abierta mas de una fraccion de segundo.
JOBS: Dict[str, dict] = {}


def _run_experience_job(job_id: str, profile: LearnerProfile, instruccion: str, contexto_extra: str, subtema: _Opt[str], transcripcion: _Opt[str] = None):
    JOBS[job_id]['status'] = 'processing'
    try:
        registro = generate_experience(profile, instruccion, contexto_extra, subtema)
        JOBS[job_id]['status'] = 'done'
        JOBS[job_id]['result'] = _experience_response(registro, transcripcion=transcripcion)
    except Exception as e:
        JOBS[job_id]['status'] = 'error'
        JOBS[job_id]['error'] = str(e)


@app.post('/experience')
def create_experience(body: ExperienceBody):
    profile = STATE.get('profile') or load_profile()
    if not profile:
        raise HTTPException(400, 'Primero crea el perfil del estudiante en /profile/interview o /profile/audio.')
    job_id = uuid.uuid4().hex[:10]
    JOBS[job_id] = {'status': 'pending', 'result': None, 'error': None}
    threading.Thread(
        target=_run_experience_job,
        args=(job_id, profile, body.instruccion, body.contexto_extra, body.subtema),
        daemon=True,
    ).start()
    return {'job_id': job_id, 'status': 'pending', 'poll_url': f'/experience/status/{job_id}'}


@app.post('/experience/audio')
async def create_experience_audio(
    audio: UploadFile = File(...),
    contexto_extra: str = Form(''),
    subtema: _Opt[str] = Form(None),
):
    profile = STATE.get('profile') or load_profile()
    if not profile:
        raise HTTPException(400, 'Primero crea el perfil del estudiante en /profile/interview o /profile/audio.')
    path = _save_upload(audio, UPLOADS)
    transcript = transcribe_audio_safe(path)
    if not transcript.strip():
        raise HTTPException(400, 'No se pudo transcribir el audio.')
    job_id = uuid.uuid4().hex[:10]
    JOBS[job_id] = {'status': 'pending', 'result': None, 'error': None}
    threading.Thread(
        target=_run_experience_job,
        args=(job_id, profile, transcript, contexto_extra, subtema, transcript),
        daemon=True,
    ).start()
    return {'job_id': job_id, 'status': 'pending', 'poll_url': f'/experience/status/{job_id}', 'transcripcion': transcript}


@app.get('/experience/status/{job_id}')
def experience_status(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(404, 'job_id no encontrado (revisa que no se haya reiniciado el servidor).')
    return job


@app.get('/experience/historial')
def experience_historial():
    return {'experiencias': read_json(EXPERIENCES_PATH, [])}


@app.get('/experience/{experience_id}')
def experience_get(experience_id: str):
    historial = read_json(EXPERIENCES_PATH, [])
    for r in historial:
        if r['id'] == experience_id:
            return _experience_response(r)
    raise HTTPException(404, 'No existe una experiencia con ese id.')


## 13. Lanzar el servidor y exponerlo con ngrok

Esta celda:
1. Arranca `uvicorn` en un hilo en background (así el kernel de Kaggle sigue libre).
2. Abre un túnel público de ngrok hacia el puerto 8000.
3. Imprime la URL pública (`https://xxxx.ngrok-free.app`) y el link a la documentación interactiva automática (`/docs`), donde puedes probar cada endpoint desde el navegador sin escribir código.

Puedes volver a correr esta celda si el túnel se cae; cierra los túneles previos automáticamente.

In [4]:
!fuser -k 8000/tcp

In [16]:
import time
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

# Cierra tuneles previos por si vuelves a correr esta celda
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass

conf.get_default().auth_token = NGROK_TOKEN
PORT = 8000


In [17]:

public_tunnel = ngrok.connect(PORT, 'http')
PUBLIC_URL = public_tunnel.public_url
print('Servidor publico (ngrok):', PUBLIC_URL)
print('Documentacion interactiva:', PUBLIC_URL + '/docs')

def _run():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()

# OPTIMIZACION: watchdog que reconecta el tunel de ngrok solo si se cae, en vez
# de que tengas que darte cuenta y volver a correr esta celda a mano. Revisa
# cada 30s; si no hay ningun tunel activo, abre uno nuevo y actualiza PUBLIC_URL
# (imprime la nueva URL para que la copies si cambio).
def _tunnel_watchdog():
    global PUBLIC_URL
    while True:
        time.sleep(30)
        try:
            if not ngrok.get_tunnels():
                print('[watchdog] Tunel de ngrok caido, reconectando...')
                nuevo = ngrok.connect(PORT, 'http')
                PUBLIC_URL = nuevo.public_url
                print('[watchdog] Nueva URL publica:', PUBLIC_URL, '| docs:', PUBLIC_URL + '/docs')
        except Exception as e:
            print('[watchdog] Error revisando/reconectando el tunel:', e)

watchdog_thread = threading.Thread(target=_tunnel_watchdog, daemon=True)
watchdog_thread.start()

print('Uvicorn corriendo en background. Deja esta celda/kernel activo para mantener el servidor vivo.')
print('El tunel se auto-reconecta solo si se cae (revisa la consola cada ~30s).')


Servidor publico (ngrok): https://gatherer-nibble-deletion.ngrok-free.dev
Documentacion interactiva: https://gatherer-nibble-deletion.ngrok-free.dev/docs
Uvicorn corriendo en background. Deja esta celda/kernel activo para mantener el servidor vivo.


INFO:     Started server process [908]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     189.217.101.162:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     189.217.101.162:0 - "GET /health HTTP/1.1" 200 OK
INFO:     189.217.101.162:0 - "OPTIONS /profile/audio HTTP/1.1" 200 OK


[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface

INFO:     189.217.101.162:0 - "POST /profile/audio HTTP/1.1" 200 OK
INFO:     189.217.101.162:0 - "OPTIONS /notes/images HTTP/1.1" 200 OK
INFO:     189.217.101.162:0 - "POST /notes/images HTTP/1.1" 200 OK
INFO:     189.217.101.162:0 - "GET /files/media/tts_notas_fa5231ac.mp3 HTTP/1.1" 206 Partial Content
INFO:     189.217.101.162:0 - "GET /files/media/tts_notas_fa5231ac.mp3 HTTP/1.1" 206 Partial Content
INFO:     189.217.101.162:0 - "OPTIONS /experience/audio HTTP/1.1" 200 OK


[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


INFO:     189.217.101.162:0 - "POST /experience/audio HTTP/1.1" 200 OK


## 14. Cliente de ejemplo

Este bloque es solo referencia — cópialo a tu máquina/proyecto local (no depende de Kaggle) para consumir la API. Reemplaza `BASE_URL` por la URL de ngrok que imprimió la celda anterior.

In [ ]:
# Ejemplo de cliente en Python para consumir este servidor desde fuera de Kaggle.
# Cambia BASE_URL por la URL https://xxxx.ngrok-free.app que imprimio la celda anterior.

import requests

BASE_URL = 'https://TU-URL-DE-NGROK.ngrok-free.app'

# 1) Salud del servicio
print(requests.get(f'{BASE_URL}/health').json())

# 2) Crear perfil a partir de texto de entrevista
resp = requests.post(f'{BASE_URL}/profile/interview', json={
    'texto': 'Me llamo Ana, quiero dominar calculo integral, soy nivel intermedio, '
             'prefiero explicaciones con muchos ejemplos, formato mixto audio y visual, '
             'estudio 30 minutos por sesion, me bloqueo con integrales por partes, '
             'quiero aprobar mi parcial la proxima semana.'
})
print(resp.json())

# 3) Subir un PDF con el temario
# with open('temario.pdf', 'rb') as f:
#     resp = requests.post(f'{BASE_URL}/curriculum/pdf', files={'pdf': f})
# print(resp.json())

# 4) Generar una leccion para el tema 0 del temario
# resp = requests.post(f'{BASE_URL}/lesson', json={'topic_index': 0, 'nivel': 'intermedio', 'enfoque': 'con ejemplos numericos'})
# leccion = resp.json()
# print(leccion['titulo'])

# 5) Descargar el audio de la leccion
# subtema = leccion['subtema']
# audio = requests.get(f'{BASE_URL}/lesson/{subtema}/audio')
# open('leccion.mp3', 'wb').write(audio.content)

# 6) Verificar el micro-reto final
# resp = requests.post(f'{BASE_URL}/lesson/{subtema}/quiz/verify', json={'respuesta_index': 1})
# print(resp.json())

# 7) Pregunta libre a Gemma
# resp = requests.post(f'{BASE_URL}/ask', json={'pregunta': 'Explica la regla de la cadena con un ejemplo corto.'})
# print(resp.json()['respuesta'])

## 15. Documentacion completa de endpoints (input / output)

Se agrega al final del notebook, como referencia rapida de todo lo que expone el servidor: lo que ya existia (sin cambios de comportamiento) y lo nuevo/modificado en esta version.

### Sin cambios (ya funcionaban asi)

- **GET `/health`** -> Sin input. Output: `{status, model, gpu}`.
- **GET `/profile`** -> Sin input. Output: `LearnerProfile` (ver mas abajo, ahora con 2 campos nuevos).
- **GET `/profile/preguntas`** -> Sin input. Output: `{preguntas: [str]}`.
- **POST `/curriculum/pdf`** -> Input: form-data `pdf` (archivo). Output: `{materia: Materia, topics: [{index, tema, subtema}]}`.
- **GET `/curriculum/topics`** -> Sin input. Output: `{topics: [{index, tema, subtema}]}`.
- **POST `/lesson`** -> Input JSON `{topic_index, nivel, enfoque}`. Output: `Leccion` completa.
- **GET `/lesson/{subtema}`** -> Sin input (path param `subtema`). Output: `Leccion`.
- **POST `/lesson/{subtema}/feedback`** -> Input JSON `{comentario}`. Output: `Leccion` actualizada (version +1).
- **POST `/lesson/{subtema}/enrich-with-notes`** -> Sin body (usa apuntes ya subidos con `/notes/images`). Output: `Leccion` actualizada.
- **GET `/lesson/{subtema}/audio`** -> Sin input. Output: archivo `audio/mpeg`.
- **GET `/lesson/{subtema}/html`** -> Sin input. Output: archivo `text/html` con micro-retos interactivos.
- **GET `/lesson/{subtema}/visual`** -> Sin input. Output: archivo `text/html` con grafico Plotly.
- **GET `/lesson/{subtema}/preview-image`** -> Sin input. Output: imagen `image/png`.
- **POST `/lesson/{subtema}/quiz/verify`** -> Input JSON `{respuesta_index}`. Output: `{correcto, explicacion}`.
- **POST `/lesson/{subtema}/repaso`** -> Sin body. Output: `{preguntas_repaso: str}`.
- **POST `/notes/images`** -> Input: form-data `imagenes` (una o mas). Output: `{analisis: {resumen, conceptos, dudas, texto_fuente}, audio_confirmacion: path|null}`.
- **POST `/progress`** -> Input JSON `{subtema, estado}`. Output: `{ok: true}`.
- **GET `/progress`** -> Sin input. Output: diccionario `{subtema: {status, updated_at}}`.
- **POST `/favorites`** -> Input JSON `{subtema, motivo}`. Output: `{ok: true}`.
- **GET `/favorites`** -> Sin input. Output: `{favoritos: [...]}`.
- **POST `/export`** -> Input JSON `{subtema, formato, contenido?}`. Output: archivo `text/plain` (`.txt` o `.tex`).
- **POST `/audio/transcribe`** -> Input: form-data `audio`. Output: `{texto: str}`.
- **POST `/tts`** -> Input: form field `texto`. Output: archivo `audio/mpeg`.
- **POST `/ask`** -> Input JSON `{pregunta, contexto}`. Output: `{respuesta: str}`.

### Modificado: perfil del estudiante

El modelo `LearnerProfile` y el prompt `PROFILE_SYSTEM` cambiaron para que un audio corto o incompleto igual genere un perfil usable, priorizando siempre el nombre:

```json
{
  "nombre": "str (unico campo realmente critico para personalizar)",
  "objetivo": "str, puede venir vacio",
  "nivel_autopercibido": "principiante|intermedio|avanzado",
  "estilo_preferido": "str, puede venir vacio",
  "formato_preferido": "str, puede venir vacio",
  "duracion_sesion": "str, puede venir vacio",
  "puntos_de_bloqueo": ["str"],
  "motivacion": "str, puede venir vacio",
  "tono_preferido": "str",
  "preferencias_extra": "str",
  "grupo_etario": "nino|adolescente|adulto|adulto_mayor (NUEVO)",
  "necesidades_especiales": "str (NUEVO, ej. letras grandes)"
}
```

- **POST `/profile/interview`** (sin cambios de firma) -> Input JSON `{texto}`. Output: `LearnerProfile` (con los 2 campos nuevos).
- **POST `/profile/audio`** (sin cambios de firma, mejor tolerancia a datos incompletos) -> Input: form-data `audio` + campo opcional `texto_extra`. Output: `{perfil: LearnerProfile, transcripcion: str}`.

### Nuevo: generacion de experiencias personalizadas

Gemma 4 decide el tipo de experiencia (cuento, lectura guiada, preguntas de repaso, explicacion simple, etc.) y escribe su propio HTML autocontenido, ajustado al perfil activo (interfaz muy simple si `grupo_etario` es `adulto_mayor`). Usa como contexto el temario/PDF cargado y los apuntes de fotos ya analizados, si existen.

- **POST `/experience`**
  - Input JSON:
    ```json
    {
      "instruccion": "str, obligatorio. Ej: 'escribeme un cuento sobre el tema de hoy'",
      "contexto_extra": "str, opcional",
      "subtema": "str, opcional. Si se pasa, se usa la leccion ya generada de ese subtema como fuente extra"
    }
    ```
  - Output JSON:
    ```json
    {
      "id": "str",
      "tipo_experiencia": "str, ej. 'cuento'",
      "titulo": "str",
      "subtema": "str",
      "contenido_texto": "str, el texto completo generado",
      "url_html": "/files/html/experiencia_xxx.html (pagina que Gemma escribio)",
      "url_audio": "/files/media/tts_xxx.mp3 o null",
      "grupo_etario_usado": "str",
      "timestamp": "str ISO"
    }
    ```
  - Requiere que ya exista un perfil (`/profile/interview` o `/profile/audio`); si no, responde 400.

- **POST `/experience/audio`**
  - Input: form-data `audio` (archivo, obligatorio) + `contexto_extra` (opcional) + `subtema` (opcional).
  - Output: igual que `/experience`, agregando `"transcripcion": "str"` con lo que dijo el estudiante.

- **GET `/experience/historial`**
  - Sin input. Output: `{"experiencias": [registro, registro, ...]}` con todas las experiencias generadas.

- **GET `/experience/{experience_id}`**
  - Sin input (path param). Output: igual formato que `/experience`. 404 si no existe ese id.

### Resumen de lo tocado vs. lo nuevo

- **Modificado** (con proposito, nada mas): `LearnerProfile` (campos opcionales + `grupo_etario` / `necesidades_especiales`), `PROFILE_SYSTEM` (prompt actualizado), tabla markdown de endpoints.
- **Nuevo**: `EXPERIENCE_SYSTEM`, `EXPERIENCES_PATH`, `_collect_apuntes_context`, `_parse_experience_blocks`, `generate_experience`, endpoints `/experience`, `/experience/audio`, `/experience/historial`, `/experience/{experience_id}`.
- **Sin tocar**: todo lo demas (modelo Gemma, ASR, lecciones, PDF, imagenes, TTS, progreso, favoritos, export, ngrok, cliente de ejemplo).


## 15. Cambios de esta version (correcciones pedidas)

Se corrigieron especificamente los 3 puntos que reportaste, sin tocar nada del resto del backend
(modelo, ASR, PDF, lecciones, TTS, progreso, favoritos, export, ngrok):

### 1. Foto -> Gemma lee -> texto natural + audio para el frontend
- `NOTES_SYSTEM` ahora pide explicitamente un campo **`respuesta_natural`**: 2-5 frases en primera
  persona, en espanol hablado, pensadas para leerse en voz alta (antes solo devolvia campos tecnicos
  como `resumen`/`conceptos`, nada pensado para sonar humano).
- `extract_notes_from_images` ya no truena el endpoint con un 500 si Gemma no devuelve JSON perfecto:
  usa el texto crudo como respaldo y deja loggeado el motivo, para que siempre haya algo que mostrar.
- `POST /notes/images` ahora regresa `respuesta_natural` como campo de primer nivel y genera el
  audio de confirmacion a partir de **ese texto real de Gemma** (antes era un prefijo fijo tipo
  'Se cargo la foto. ' + resumen, que no era una respuesta natural).

### 2. Perfil: por que no extraia el nombre / info
- Bug real: `LearnerProfile(**extract_json_block(raw))` combinado con un `extract_json_block` que, si
  fallaba al parsear, dejaba que Pydantic rellenara todo con sus valores por defecto (`nombre='Estudiante'`,
  listas vacias, etc.) **sin ningun error visible**. Es decir, cuando Gemma no seguia el formato al pie
  de la letra, el sistema lo disimulaba en vez de fallar o avisar.
- `extract_json_block` ahora es explicito: si no encuentra JSON valido, lanza `ValueError` con la
  salida cruda de Gemma en el mensaje (queda en logs), y si recibe `required_keys` avisa por consola
  cuando Gemma dejo campos clave vacios.
- `PROFILE_SYSTEM` se reescribio con instrucciones mas estrictas (nada de texto fuera del JSON, nada
  de '...' literal, prioridad explicita al nombre) y un ejemplo completo de entrada/salida para que
  el modelo tenga una referencia clara del formato esperado.
- Se agrego una red de seguridad minima (`_regex_fallback_nombre`) que **solo actua si Gemma devolvio
  'nombre' vacio**: busca patrones tipo 'me llamo X' / 'mi nombre es X' en el texto original. No
  reemplaza ni compite con lo que Gemma ya extrajo, es unicamente para el caso en que vino vacio.

### 3. Generacion de experiencias: por que no generaba lo esperado
- `_parse_experience_blocks` usaba un regex 'perezoso' que **requeria encontrar el marcador de cierre**
  (`===FIN===`) para capturar el HTML. Como el HTML completo facilmente supera el limite de
  `max_new_tokens`, la generacion se cortaba antes de escribir `===FIN===` y el parser perdia *todo*
  el HTML que Gemma si alcanzo a escribir, cayendo siempre al respaldo generico.
- Se reescribio el parseo para capturar cada bloque por indice de texto (si no hay marcador de cierre,
  toma hasta el final en vez de fallar), y se agrego limpieza de code-fences (` ```html `) por si Gemma
  envuelve el HTML en un bloque de codigo a pesar de la instruccion.
- `generate_experience` subio `max_new_tokens` de 4000 a 6000 y bajo `temperature` de 0.55 a 0.45
  para reducir la probabilidad de cortes y mejorar la adherencia al formato de marcadores.
- `EXPERIENCE_SYSTEM` ahora deja explicito: cada marcador en su propia linea, nada de code-fences
  alrededor del HTML, y recordatorio de escribir siempre `===FIN===` al final.
- Se agregaron `print()` de diagnostico (HTML cortado, contenido_texto vacio, respaldo generico usado)
  para que si Gemma sigue sin seguir el formato, sea visible en los logs de Kaggle, en vez de fallar en
  silencio.

Ningun endpoint cambio de firma (mismos inputs/outputs documentados arriba); `POST /notes/images` solo
gano el campo adicional `respuesta_natural` en la respuesta.


## 16. Optimizaciones aplicadas en esta version (velocidad + estabilidad de ngrok)

**Problema real detectado:** en `/experience`, Gemma tenia que redactar contenido Y una pagina HTML
completa autocontenida en la misma generacion (`max_new_tokens=6000`). Eso son varios minutos de
generacion token a token, y mientras el request HTTP seguia abierto esperando esa respuesta, el tunel
de ngrok se caia (ninguna conexion aguanta bloqueada tanto tiempo sin datos).

**Cambios:**
1. **Gemma ya no escribe HTML** (celdas 13 y 27). Solo redacta `TIPO / TITULO / TEXTO / AUDIO`; el HTML
   se arma en Python (`_render_experience_html`) respetando el perfil (letras grandes para
   `adulto_mayor`, tono calido para `nino`, etc.). `max_new_tokens` bajo de 6000 a 1400.
2. **`/experience` y `/experience/audio` son asincronos** (celda 31): responden de inmediato con un
   `job_id` y la generacion corre en un hilo en background. El cliente consulta el resultado con
   `GET /experience/status/{job_id}` (status: `pending` -> `processing` -> `done`/`error`) cada 2-3
   segundos. Ninguna conexion HTTP/ngrok queda abierta mas de una fraccion de segundo.
3. **`attn_implementation='sdpa'`** al cargar el modelo (celda 7) y `torch.inference_mode()` en vez de
   `no_grad` (celda 9): generacion algo mas rapida sin cambiar la calidad de las respuestas.
4. **Watchdog de ngrok** (celda 35): revisa cada 30s si el tunel sigue vivo y lo reconecta solo si se cae,
   sin que tengas que volver a correr la celda a mano.

**Como consumir `/experience` ahora desde tu cliente:**
```python
resp = requests.post(f'{BASE_URL}/experience', json={'instruccion': 'cuentame un cuento sobre esto'})
job_id = resp.json()['job_id']

import time
while True:
    estado = requests.get(f'{BASE_URL}/experience/status/{job_id}').json()
    if estado['status'] == 'done':
        experiencia = estado['result']
        break
    elif estado['status'] == 'error':
        raise RuntimeError(estado['error'])
    time.sleep(2)
```

**Si sigue sintiendose lento**, los siguientes candidatos (no aplicados aqui para no arriesgar calidad sin que
lo pidas) son bajar `max_new_tokens` de `/lesson` (3600) y `/lesson/{subtema}/feedback` (3200), que son los
otros dos endpoints con generaciones largas.
